<a href="https://colab.research.google.com/github/Moriarty7819/Gold-price-prediction/blob/main/Gold_Price.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import tensorflow as tf
from keras import Model
from keras.layers import Input, Dense, Dropout
from keras.layers import LSTM

In [3]:
df = pd.read_csv('/content/Gold Futures Historical Data.csv')

In [4]:
df["Date"] = df["Date"].astype(str).str.replace("-", "/")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Date"] = df["Date"].dt.strftime("%m/%d/%Y")

In [5]:
df

,Date,Price,Open,High,Low,Vol.,Change %
0,01/03/2005,429.7,438.9,438.9,427.8,82850,-1.98
1,01/04/2005,429.2,430.4,431.3,424.8,54830,-0.12
2,01/05/2005,427.3,429,429.3,425.9,43720,-0.44
3,01/06/2005,421.6,427,428.3,421.1,65180,-1.33
4,01/07/2005,419.5,422.1,425.8,417.2,97970,-0.50
...,...,...,...,...,...,...,...
4992,02/18/2026,"5,009.50","4,898.50","5,031.90","4,868.50",98200,2.11
4993,02/19/2026,"4,997.40","4,993.70","5,042.80","4,971.50",87030,-0.24
4994,02/20/2026,"5,080.90","5,015.00","5,131.00","4,999.30",136740,1.67
4995,02/23/2026,"5,207.50","5,114.00","5,236.70","5,103.20",1620,2.49


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      4997 non-null   object 
 1   Price     4997 non-null   object 
 2   Open      4997 non-null   object 
 3   High      4997 non-null   object 
 4   Low       4997 non-null   object 
 5   Vol.      4997 non-null   int64  
 6   Change %  4997 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 273.4+ KB


In [7]:
df.drop(['Vol.', 'Change %'], axis=1, inplace=True)


In [8]:
NumCols = df.columns.drop(['Date'])
df[NumCols] = df[NumCols].replace({',': ''}, regex=True)
df[NumCols] = df[NumCols].astype('float64')


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    4997 non-null   object 
 1   Price   4997 non-null   float64
 2   Open    4997 non-null   float64
 3   High    4997 non-null   float64
 4   Low     4997 non-null   float64
dtypes: float64(4), object(1)
memory usage: 195.3+ KB


In [10]:
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values(by='Date', ascending=True, inplace=True)
df.reset_index(drop=True, inplace=True)


In [11]:
df

,Date,Price,Open,High,Low
0,2005-01-03,429.7,438.9,438.9,427.8
1,2005-01-04,429.2,430.4,431.3,424.8
2,2005-01-05,427.3,429.0,429.3,425.9
3,2005-01-06,421.6,427.0,428.3,421.1
4,2005-01-07,419.5,422.1,425.8,417.2
...,...,...,...,...,...
4992,2026-02-18,5009.5,4898.5,5031.9,4868.5
4993,2026-02-19,4997.4,4993.7,5042.8,4971.5
4994,2026-02-20,5080.9,5015.0,5131.0,4999.3
4995,2026-02-23,5207.5,5114.0,5236.7,5103.2


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4997 entries, 0 to 4996
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    4997 non-null   datetime64[ns]
 1   Price   4997 non-null   float64       
 2   Open    4997 non-null   float64       
 3   High    4997 non-null   float64       
 4   Low     4997 non-null   float64       
dtypes: datetime64[ns](1), float64(4)
memory usage: 195.3 KB


In [13]:
df

,Date,Price,Open,High,Low
0,2005-01-03,429.7,438.9,438.9,427.8
1,2005-01-04,429.2,430.4,431.3,424.8
2,2005-01-05,427.3,429.0,429.3,425.9
3,2005-01-06,421.6,427.0,428.3,421.1
4,2005-01-07,419.5,422.1,425.8,417.2
...,...,...,...,...,...
4992,2026-02-18,5009.5,4898.5,5031.9,4868.5
4993,2026-02-19,4997.4,4993.7,5042.8,4971.5
4994,2026-02-20,5080.9,5015.0,5131.0,4999.3
4995,2026-02-23,5207.5,5114.0,5236.7,5103.2


In [14]:
df.duplicated().sum()


np.int64(0)

In [15]:
df.isnull().sum().sum()


np.int64(0)

In [16]:
test_size = df[df.Date.dt.year.isin([2020,2021,2022,2023,2024, 2025])].shape[0]
test_size

1322

In [17]:
scaler = MinMaxScaler()
scaler.fit(df.Price.values.reshape(-1,1))

MinMaxScaler()

In [18]:
window_size = 60


In [19]:
train_data = df.Price[:-test_size]
train_data = scaler.transform(train_data.values.reshape(-1,1))

In [20]:
X_train = []
y_train = []

for i in range(window_size, len(train_data)):
    X_train.append(train_data[i-60:i, 0])
    y_train.append(train_data[i, 0])

In [21]:
test_data = df.Price[-test_size-60:]
test_data = scaler.transform(test_data.values.reshape(-1,1))

In [22]:
X_test = []
y_test = []

for i in range(window_size, len(test_data)):
    X_test.append(test_data[i-60:i, 0])
    y_test.append(test_data[i, 0])

In [23]:
X_train = np.array(X_train)
X_test  = np.array(X_test)
y_train = np.array(y_train)
y_test  = np.array(y_test)

X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
X_test  = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
y_train = np.reshape(y_train, (-1,1))
y_test  = np.reshape(y_test, (-1,1))

In [24]:
print('X_train Shape: ', X_train.shape)
print('y_train Shape: ', y_train.shape)
print('X_test Shape:  ', X_test.shape)
print('y_test Shape:  ', y_test.shape)

X_train Shape:  (3615, 60, 1)
y_train Shape:  (3615, 1)
X_test Shape:   (1322, 60, 1)
y_test Shape:   (1322, 1)


In [25]:
from tensorflow.keras.callbacks import EarlyStopping

In [26]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [27]:
def define_model2():
    input1 = Input(shape=(window_size,1))
    x = LSTM(units = 64, return_sequences=True)(input1)
    x = Dropout(0.2)(x)
    x = LSTM(units = 64, return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(units = 64)(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='softmax')(x)
    dnn_output = Dense(1)(x)

    model = Model(inputs=input1, outputs=[dnn_output])
    model.compile(loss='mean_squared_error', optimizer='Nadam')
    model.summary()

    return model

In [28]:
def define_model():
    input1 = Input(shape=(window_size,1))
    x = LSTM(units = 64, return_sequences=True)(input1)
    x = Dropout(0.2)(x)
    x = LSTM(units = 64, return_sequences=True)(x)
    x = Dropout(0.2)(x)
    x = LSTM(units = 64)(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    dnn_output = Dense(1)(x)

    model = Model(inputs=input1, outputs=[dnn_output])
    model.compile(loss='mean_squared_error', optimizer='Nadam')
    model.summary()

    return model

In [29]:
model = define_model()
history = model.fit(X_train, y_train, epochs=150, batch_size=32, validation_split=0.1, verbose=1)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 60, 1)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 60, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 60, 64)         │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,057 (332.25 KB)

 Trainable params: 85,057 (332.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 17s 102ms/step - loss: 0.0011 - val_loss: 3.0153e-04
Epoch 2/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 97ms/step - loss: 1.9746e-04 - val_loss: 7.5299e-05
Epoch 3/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 97ms/step - loss: 1.5374e-04 - val_loss: 6.8966e-05
Epoch 4/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 101ms/step - loss: 1.2910e-04 - val_loss: 5.4964e-05
Epoch 5/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 20s 100ms/step - loss: 1.1565e-04 - val_loss: 4.1999e-05
Epoch 6/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - loss: 1.0499e-04 - val_loss: 3.5209e-05
Epoch 7/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 96ms/step - loss: 9.6482e-05 - val_loss: 1.0992e-04
Epoch 8/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 9s 87ms/step - loss: 8.2319e-05 - val_loss: 3.0921e-05
Epoch 9/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 11s 96ms/step - loss: 8.6134e-05 - val_loss: 4.6596e-05
Epoch 10/150
102/102 ━━━━━━━━━━━━━━━━━━━━ 10s 98ms/step - loss: 7.6542e-05 - val_loss: 4.4252e-05
Epoch 11/150
102/102 ━━━━━━━━━━

In [30]:
result = model.evaluate(X_test, y_test)
y_pred = model.predict(X_test)


42/42 ━━━━━━━━━━━━━━━━━━━━ 1s 28ms/step - loss: 0.0021    
42/42 ━━━━━━━━━━━━━━━━━━━━ 2s 32ms/step


In [31]:
MAPE = mean_absolute_percentage_error(y_test, y_pred)
Accuracy = 1 - MAPE

In [32]:
print("Test Loss:", result)
print("Test MAPE:", MAPE)
print("Test Accuracy:", Accuracy)

Test Loss: 0.0020519602112472057
Test MAPE: 0.03988884797277014
Test Accuracy: 0.9601111520272299
